<div dir="rtl">

# תמלול אודיו ווידאו בעברית — Kaggle

מחברת זו מתמללת קבצי אודיו ווידאו בעברית ומפיקה קובץ טקסט וקובץ כתוביות.

**אין צורך בחשבון או בהרשמה לשום שירות חיצוני.** רק חשבון Kaggle חינמי.

היא מבוססת על [המחברת המקורית](https://github.com/Sourasky-DHLAB/Whisper) של
[הספרייה המרכזית ע"ש סוראסקי](https://cenlib.tau.ac.il/), אוניברסיטת תל אביב (עודד זרחיה),
ונכתבה מחדש ב-2026 עבור Kaggle עם מודל עברי עדכני.

| רכיב | מה בשימוש | למה |
|---|---|---|
| מנוע תמלול | [faster-whisper](https://github.com/SYSTRAN/faster-whisper) | מהיר פי ~4 מ-`openai-whisper` וצורך פחות זיכרון GPU |
| מודל | [`ivrit-ai/whisper-large-v3-turbo-ct2`](https://huggingface.co/ivrit-ai/whisper-large-v3-turbo-ct2) | Whisper שכוונן במיוחד לעברית — דיוק גבוה משמעותית מהמודל הרגיל |
| קלט | ffmpeg | מקבל **כל** פורמט אודיו/וידאו — אין צורך להמיר ל-WAV ידנית |

### רוצים לדעת מי אמר מה?

מחברת זו מתמללת בלבד ואינה מזהה דוברים. לתמלול ראיונות עם זיהוי דוברים
השתמשו ב-[hebrew-diarization.ipynb](https://github.com/ablolarof/hebrewWhisper/blob/main/Kaggle/hebrew-diarization.ipynb).
שימו לב שהיא כן דורשת חשבון Hugging Face חינמי.

</div>

---

### Setup — two toggles and an upload

**1. Turn on the GPU.** Right sidebar → **Session options** → **Accelerator** → `GPU T4 x2` or `GPU P100`.

**2. Turn on the internet.** Right sidebar → **Session options** → **Internet** → **On**.
(Kaggle requires a phone-verified account for this. Without it, the model can't download.)

**3. Upload your recordings.** Right sidebar → **Add Input** → **Upload** → **New Dataset**, named
`audiofiles`. They will appear under `/kaggle/input/audiofiles/`.

That's everything. No tokens, no accounts, no terms to accept.


<div dir="rtl">

## 1. בדיקת המעבד הגרפי

הריצו את התא הבא כדי לוודא שהוקצה GPU. אם הפלט ריק או שגוי — חזרו להגדרות והפעילו את ה-Accelerator.

</div>

In [ ]:
!nvidia-smi

<div dir="rtl">

## 2. התקנת ספריות

</div>

We deliberately **do not install or pin `torch`**. Kaggle ships a PyTorch build matched to its own CUDA
driver; overriding it is what broke the original notebook. Only `faster-whisper` is added on top.

This cell takes 1-2 minutes.

> **A long red `ERROR: pip's dependency resolver...` block here is expected. Ignore it.**
> Kaggle's image ships hundreds of packages that were already inconsistent with each other, and pip
> audits *all* of them after any install. Some entries even refer to packages that were missing before
> this notebook ran.
>
> The check that matters is the small version table printed at the end of this cell. If those lines
> appear, the install worked — pip would have stopped before reaching them otherwise.

In [ ]:
%pip install -q "faster-whisper>=1.1.0"

# Print what we actually ended up with, so version problems are visible now
# rather than as a confusing failure three cells later.
import importlib.metadata as meta
import torch
for pkg in ("torch", "faster-whisper", "ctranslate2"):
    try:
        print(f"{pkg:20s} {meta.version(pkg)}")
    except meta.PackageNotFoundError:
        print(f"{pkg:20s} NOT INSTALLED")
print(f"{'CUDA available':20s} {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"{'GPU':20s} {torch.cuda.get_device_name(0)}")
    # cuDNN is printed because a mismatch between its major version and the one
    # ctranslate2 was built against is the classic way faster-whisper fails, and
    # the error it produces names a library file rather than the real problem.
    print(f"{'cuDNN':20s} {torch.backends.cudnn.version()}")

> **If a later cell fails with an `ImportError` mentioning numpy** — for example
> `cannot import name '_center' from 'numpy._core.umath'` — use **Run → Restart Session**, then run
> the cells again from the top. Installing packages can upgrade numpy underneath a kernel that has
> already loaded the old one, leaving the two halves of numpy disagreeing. It is harmless, it only
> happens on a cold session, and restarting fixes it permanently for that session.

<div dir="rtl">

## 3. הגדרות

זהו התא היחיד שרוב המשתמשים צריכים לשנות.

</div>

| Setting | What it does |
|---|---|
| `INPUT_DIR` | Where your uploaded files live. Leave as-is if your dataset is named `audiofiles`. A single file's path also works. |
| `LANGUAGE` | `he` for Hebrew, `ar` for Arabic, `en` for English. The ivrit-ai model is Hebrew-only — switch `WHISPER_MODEL` to `large-v3` for other languages. |

In [ ]:
from pathlib import Path

# Everything worth changing lives in this one cell, so you never have to hunt
# through the notebook for a path or a model name.

# --- input / output -------------------------------------------------------
# On Kaggle, /kaggle/input is read-only and holds your uploaded Dataset.
# /kaggle/working is the only writable folder, and it is what the Output panel
# on the right offers for download.
INPUT_DIR  = Path("/kaggle/input/audiofiles")
OUTPUT_DIR = Path("/kaggle/working/transcriptions")

# --- transcription --------------------------------------------------------
# A Whisper that was further trained on Hebrew, so it is far more accurate on
# Hebrew than the standard model. That training degraded its ability to *guess*
# the language, which is why LANGUAGE is always passed explicitly rather than
# left to autodetect. For any other language, switch to "large-v3".
WHISPER_MODEL = "ivrit-ai/whisper-large-v3-turbo-ct2"
LANGUAGE      = "he"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {OUTPUT_DIR}")

<div dir="rtl">

## 4. איתור קבצי הקלט

</div>

In [ ]:
# Video formats are included on purpose: the audio is pulled out of them in the
# next cell, so an .mp4 straight off a phone works with no preparation. This set
# exists only to skip the stray .csv, .txt or .DS_Store that tends to ride along
# in an uploaded dataset.
MEDIA_EXT = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".aac", ".wma",
             ".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}

# Both checks below stop the notebook immediately with an explanation. The
# alternative is a confusing empty result several cells later, by which point the
# real cause is hard to see.
if not INPUT_DIR.exists():
    # /kaggle/input is read-only. You cannot create this folder from code - it is
    # filled in by Kaggle when you attach a Dataset. So a missing folder means
    # either no Dataset is attached, or yours is attached under a different name
    # (Kaggle turns "My Audio" into "my-audio"). Listing what is actually mounted
    # turns a dead end into an answer.
    kaggle_input = Path("/kaggle/input")
    attached = sorted(p.name for p in kaggle_input.iterdir()) if kaggle_input.exists() else []
    if attached:
        hint = (f"Datasets attached right now: {', '.join(attached)}.\n"
                "If yours is in that list, change INPUT_DIR above to match it.\n")
    else:
        hint = "No datasets are attached to this notebook yet.\n"
    raise FileNotFoundError(
        f"{INPUT_DIR} does not exist.\n" + hint +
        "To attach one: right sidebar -> Add Input -> Upload -> New Dataset, named 'audiofiles'.\n"
        "Note: /kaggle/input is read-only, so this folder cannot be created by code."
    )

# INPUT_DIR is normally a folder, but pasting the path of the single file you
# want transcribed is a natural thing to do, so accept that too.
if INPUT_DIR.is_file():
    media_files = [INPUT_DIR]
else:
    # rglob searches subfolders too, because uploading a zip keeps its folder
    # structure and the files often end up a level down rather than at the top.
    media_files = sorted(p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in MEDIA_EXT)

if not media_files:
    present = sorted(p.name for p in INPUT_DIR.rglob("*") if p.is_file())[:20]
    raise FileNotFoundError(
        f"No audio or video files found under {INPUT_DIR}\n"
        f"Files present: {', '.join(present) if present else '(the folder is empty)'}"
    )

for p in media_files:
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")
print(f"\n{len(media_files)} file(s) to transcribe.")

<div dir="rtl">

## 5. הכנת האודיו

</div>

The original notebook required you to convert everything to mono WAV by hand. This cell does it for
you with ffmpeg — **any** audio or video format works, including MP4 straight off a phone.

In [ ]:
import subprocess, shutil, tempfile

if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg not found on this machine.")

# A scratch folder, deliberately not inside /kaggle/working: these converted
# files are throwaway, and putting them in the output folder would bury your
# actual transcripts among large WAVs in the download panel.
PREPARED_DIR = Path(tempfile.mkdtemp(prefix="prepared_audio_"))

def prepare_audio(src: Path) -> Path:
    """Convert any media file to the format the model wants.

    Whisper expects 16 kHz mono and will convert anything else itself. Doing it
    once here means you never have to convert files by hand, which the original
    notebook demanded.
    """
    dst = PREPARED_DIR / (src.stem + ".wav")
    cmd = ["ffmpeg", "-y", "-loglevel", "error",
           "-i", str(src),
           "-vn",              # ignore any video stream; decoding pictures is wasted work
           "-ac", "1",         # mono: one channel is what the model reads
           "-ar", "16000",     # 16 kHz: speech models are trained at this rate
           "-c:a", "pcm_s16le",  # uncompressed, so no second round of lossy encoding
           str(dst)]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"ffmpeg failed on {src.name}:\n{proc.stderr}")
    return dst

prepared = {}
for p in media_files:
    prepared[p] = prepare_audio(p)
    dur = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(prepared[p])],
        capture_output=True, text=True).stdout.strip()
    print(f"  {p.name} -> {float(dur)/60:.1f} min")

print(f"\nPrepared {len(prepared)} file(s).")

<div dir="rtl">

## 6. טעינת המודל

הורדת המודל בפעם הראשונה אורכת מספר דקות (כ-1.6 ג'יגה-בייט).

</div>

In [ ]:
import torch
from faster_whisper import WhisperModel

# compute_type is the number format the model calculates in. float16 uses half
# the memory of the default and runs faster on a GPU, with no accuracy loss worth
# worrying about for speech. CPUs have no useful float16 support, so there we
# fall back to int8, which is cruder but at least runs.
if torch.cuda.is_available():
    device, compute_type = "cuda", "float16"
else:
    device, compute_type = "cpu", "int8"
    print("WARNING: no GPU detected. This will be extremely slow. "
          "Enable the accelerator in Session options.")

print(f"Device: {device} ({compute_type})")
print("Loading model ...")
model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute_type)
print("Model ready.")

<div dir="rtl">

## 7. פונקציות עזר

אין צורך לשנות דבר בתא הבא.

</div>

In [ ]:
def format_timestamp(seconds: float, srt: bool = False) -> str:
    """HH:MM:SS, or HH:MM:SS,mmm for SRT.

    Work in integer milliseconds throughout: float truncation otherwise renders
    2.4s as '00:00:02,399', which is off by a millisecond on every cue.
    """
    seconds = max(0.0, seconds)
    if srt:
        total_ms = int(round(seconds * 1000))
        h, rem = divmod(total_ms, 3_600_000)
        m, rem = divmod(rem, 60_000)
        s, ms = divmod(rem, 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    total = int(seconds)
    return f"{total // 3600:02d}:{(total % 3600) // 60:02d}:{total % 60:02d}"


def group_into_paragraphs(segments, max_seconds=45.0, pause_seconds=2.0):
    """Merge Whisper's segments into readable paragraphs.

    Whisper emits a segment every few seconds - 68 of them for a 3.5 minute
    recording - so one line each gives a choppy wall of fragments. Merging them
    reads far better. A new paragraph starts after a real pause in the speech,
    which usually lines up with a change of subject, or once a paragraph has run
    long enough that it needs a break regardless.
    """
    paragraphs = []
    for seg in segments:
        text = (seg.text or "").strip()
        if not text:
            continue
        if not paragraphs:
            start_new = True
        else:
            last = paragraphs[-1]
            gap = seg.start - last["end"]
            running = last["end"] - last["start"]
            start_new = gap >= pause_seconds or running >= max_seconds
        if start_new:
            paragraphs.append({"start": seg.start, "end": seg.end, "text": text})
        else:
            paragraphs[-1]["end"] = seg.end
            paragraphs[-1]["text"] += " " + text
    return paragraphs

print("Helpers defined.")

<div dir="rtl">

## 8. תמלול

זהו התא הכבד. כשעה של אודיו לוקחת בערך 4-6 דקות על T4.

</div>

In [ ]:
import time, gc

results = {}

for original, wav in prepared.items():
    print(f"\n{'=' * 70}\n{original.name}\n{'=' * 70}")
    t0 = time.time()

    segments_gen, info = model.transcribe(
        str(wav),
        language=LANGUAGE,
        vad_filter=True,                       # skip long silences
        vad_parameters={"min_silence_duration_ms": 500},
        # No word_timestamps here. The diarization notebook needs them to match
        # words against speaker turns; with no speakers to match, asking for them
        # would only cost time.
    )
    segments = list(segments_gen)              # generator -> list (this is where the work happens)

    if not segments:
        print("  No speech detected - skipping.")
        continue

    paragraphs = group_into_paragraphs(segments)
    results[original] = {"segments": segments, "paragraphs": paragraphs}

    words = sum(len(s.text.split()) for s in segments)
    print(f"  {len(segments)} segments, ~{words} words, "
          f"{len(paragraphs)} paragraphs ({time.time() - t0:.0f}s)")

    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

print(f"\n\nTranscribed {len(results)} file(s).")

<div dir="rtl">

## 9. שמירת התמלילים

הקבצים נשמרים תחת `/kaggle/working/transcriptions/`. להורדה: לחצו על **Output** בסרגל הימני.

</div>

In [ ]:
# U+200F, the right-to-left mark. It is invisible, but without it an editor that
# defaults to left-to-right will mangle a line that mixes Hebrew with digits or
# Latin letters - which every line here does, because of the timestamps.
RLM = "\u200f"

def write_txt(path, paragraphs):
    with open(path, "w", encoding="utf-8") as f:
        for p in paragraphs:
            f.write(f"\n{RLM}[{format_timestamp(p['start'])}]\n")
            f.write(f"{RLM}{p['text']}\n")


def write_srt(path, segments):
    """Subtitle file: one cue per Whisper segment.

    Cues follow Whisper's segments rather than the paragraphs of the txt file.
    A paragraph can run for 45 seconds, which reads well on a page but is far
    too long to sit on screen as a subtitle.
    """
    with open(path, "w", encoding="utf-8") as f:
        idx = 0
        for seg in segments:
            text = (seg.text or "").strip()
            if not text:
                continue
            idx += 1
            f.write(f"{idx}\n")
            f.write(f"{format_timestamp(seg.start, srt=True)} --> "
                    f"{format_timestamp(seg.end, srt=True)}\n")
            f.write(f"{RLM}{text}\n\n")


written = []
for original, data in results.items():
    txt_path = OUTPUT_DIR / f"{original.stem}.txt"
    write_txt(txt_path, data["paragraphs"])
    written.append(txt_path)

    srt_path = OUTPUT_DIR / f"{original.stem}.srt"
    write_srt(srt_path, data["segments"])
    written.append(srt_path)

for p in written:
    print(f"  {p}  ({p.stat().st_size / 1024:.1f} KB)")
print(f"\nWrote {len(written)} file(s).")

<div dir="rtl">

## 10. תצוגה מקדימה

</div>

In [ ]:
from IPython.display import HTML, display
import html as html_lib

PREVIEW_PARAGRAPHS = 10

for original, data in results.items():
    rows = []
    for p in data["paragraphs"][:PREVIEW_PARAGRAPHS]:
        rows.append(
            f'<div style="margin-bottom:0.9em">'
            f'<span style="color:#888;font-size:0.85em">'
            f'[{format_timestamp(p["start"])}]</span><br>'
            f'{html_lib.escape(p["text"])}</div>'
        )
    more = ""
    if len(data["paragraphs"]) > PREVIEW_PARAGRAPHS:
        more = (f'<div style="color:#888">... ועוד '
                f'{len(data["paragraphs"]) - PREVIEW_PARAGRAPHS} פסקאות</div>')
    display(HTML(
        f'<div dir="rtl" style="text-align:right;font-size:1.05em;'
        f'line-height:1.6;font-family:Arial,sans-serif">'
        f'<h3>{html_lib.escape(original.name)}</h3>'
        f'{"".join(rows)}{more}</div>'
    ))

---

<div dir="rtl">

## פתרון תקלות

</div>

| Symptom | Cause and fix |
|---|---|
| `ImportError` mentioning numpy | Installing upgraded numpy under a running kernel. **Run → Restart Session**, then run from the top. Happens once per cold session. |
| `/kaggle/input/audiofiles does not exist` | The Dataset was never attached, or Kaggle renamed it (`My Audio` becomes `my-audio`). The error lists what *is* attached — point `INPUT_DIR` at that. |
| `Could not load library libcudnn_ops.so` | cuDNN mismatch in the Kaggle image. Set `compute_type = "int8_float16"` in the model-loading cell. |
| No GPU detected | Session options → Accelerator → GPU. Without it this runs, but far too slowly to be useful. |
| Model download fails | Session options → Internet → On. Requires a phone-verified Kaggle account. |
| Session dies partway | Kaggle caps sessions at 12h. Process fewer files per run. |
| Hebrew displays backwards in Notepad | Open it in an RTL-aware editor (Word, VS Code, Google Docs). The output carries invisible right-to-left marks, which most viewers honour; Notepad does not. |
| Transcription is poor quality | Confirm `LANGUAGE = "he"`. The ivrit-ai model is Hebrew-only — for other languages switch `WHISPER_MODEL` to `"large-v3"`. |
| You need to know who said what | Use [hebrew-diarization.ipynb](https://github.com/ablolarof/hebrewWhisper/blob/main/Kaggle/hebrew-diarization.ipynb) instead. It adds speaker labels, at the cost of needing a free Hugging Face account. |

<div dir="rtl">

### קרדיטים

מבוסס על המחברת המקורית של [הספרייה המרכזית ע"ש סוראסקי](https://cenlib.tau.ac.il/), אוניברסיטת תל אביב.
מודל התמלול בעברית: [ivrit.ai](https://www.ivrit.ai/).

</div>